# 01 · AKUDENTAL Panoramic WSP Benchmark (Validation Only)

Thin reproduction and Study 2 notebook importing `wsp_core` and `benchmarks.akudental`.

- **Track 2 (`akudental_seg`):** Reproduces Study 1 (`pano-wsp82-v1`, 6/6 PASS gate), runs A/B/C ablations, paired swaps, inside/outside FP, boundary F1, edge recall, and **excluded annotated tooth-pixel rate**.
- **Track 1 (`akudental_cls`):** Radiopaque restoration / implant detection (`cls-lenet2d-v1`, AUC + sign-flip & margin-crop checks).
- **3-Arm Label-Budget Curves:** Frozen WSP (`K = 0`), Random-Init Adam, WSP Warm-Start Adam, and Trained Position-Only.

**Governance:** Held-out test set remains sealed (`split='val'`). Keep all private radiograph overlays inside this Colab runtime and clear outputs before sharing.

In [ ]:
# Install `dental-wsp` in Colab (point REPO_DIR to your cloned repo or Drive path)
from pathlib import Path
import sys

REPO_DIR = Path("/content/dental-wsp")  # Update if mounted from Drive
if REPO_DIR.is_dir():
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
drive.mount("/content/drive", readonly=True)
WORK = Path("/content/drive/MyDrive/Colab Notebooks/WSP_Work")

from wsp_core.guards import exposure_guard, private_errors
exposure_guard(WORK)
print("Drive mounted read-only and exposure_guard passed.")

In [ ]:
import pandas as pd
from benchmarks.akudental.adapter import load_akudental_seg_zip
from wsp_core.evaluation.segmentation import (
    evaluate_segmentation_paired_audit,
    evaluate_segmentation_swaps,
)
from wsp_core.schemas import strict_json_loads, validate_parameters

ARCHIVE = WORK / "AKUDENTAL_PANORAMIC_INIT_V1.zip"
SPLIT_FILE = WORK / "AKUDENTAL_Part1_Runs/akudental-controlled-init-part1-v2/split_plan.private.json"
RESPONSES_LOCK = WORK / "AKUDENTAL_Whole_Model_WSP_v1/responses.lock.private.json"

with private_errors("Failed to load AKUDENTAL validation cohort or locked replies."):
    val_samples = load_akudental_seg_zip(ARCHIVE, SPLIT_FILE, split="val")
    lock = strict_json_loads(RESPONSES_LOCK.read_bytes())

rows = []
for slot, rec in lock["attempts"].items():
    if rec.get("status") != "valid":
        continue
    payload = rec["parameters"]
    validate_parameters(payload, expected_architecture="pano-wsp82-v1")
    swaps = evaluate_segmentation_swaps(payload, val_samples)
    audit = evaluate_segmentation_paired_audit(payload, val_samples)
    rows.append({
        "Reply": slot,
        "Intact Dice (%)": swaps["intact"]["correct_dice_pct"],
        "Swap loss (pp)": swaps["intact"]["swap_loss_pp"],
        "Outside FP share (%)": audit["outside_fp_reduction_share_pct"],
        "Delta edge recall (pp)": audit["edge_recall_gain_pp"],
        "Box IoU A (%)": audit["box_iou_a_pct"],
        "Excluded tooth pixels A (%)": audit["excluded_tooth_a_pct"],
    })

display(pd.DataFrame(rows).set_index("Reply").round(4))

In [ ]:
# Study 2 (requires PyTorch + prespecified train split; test remains sealed):
# from wsp_core.training.trainer import TrainingConfig, sample_nested_group_budgets, train_segmentation_budget
# train_samples = load_akudental_seg_zip(ARCHIVE, SPLIT_FILE, split="train")
# budgets = sample_nested_group_budgets([s.group_id for s in train_samples], [1, 2, 4, 8, 16, 32], seed=0)
print("Configure and authorize Study 2 budgets before running training.")